# Valence Band Offset (VBO)

Calculate the valence band offset for a labeled interface using a DFT workflow on the Mat3ra platform.

The notebook supports two modes controlled by `IS_POLAR` in cell 1.2:

- `IS_POLAR = False`: run the standard VBO workflow and report the workflow VBO together with the average ESP profiles.
- `IS_POLAR = True`: keep the same workflow VBO calculation, then run an additional notebook-side post-process at the end that fits the interface and bulk ESP profiles over the slab regions and plots the polar fit.

When the polar option is enabled, the workflow itself is unchanged. The extra polar analysis is performed in the final notebook cells using the saved interface, left-slab, and right-slab materials plus the ESP results returned by the job.

<h2 style="color:green">Usage</h2>

1. Create and save an interface with labels (for example via `made`).
1. Set the interface and calculation parameters in cells 1.2 and 1.3 below, including `IS_POLAR` if the interface is polar.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the VBO result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for the interface, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create materials: load an interface from the `../uploads` folder, split it into interface/left/right parts using interface labels, strip labels required by Quantum ESPRESSO, and save all three materials to the platform tagged with the interface name.
1. Configure workflow: select application, load the VBO workflow from Standata, assign materials to subworkflows by role, set model and computational parameters, and preview the workflow.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with materials and workflow configuration: assemble the job from materials, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the workflow valence band offset and average ESP profiles.
1. If `IS_POLAR = True`, run the final polar post-process cells to fit the ESP profile and generate the polar-fit plot in the notebook.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install('mat3ra-notebooks-utils')
    from mat3ra.notebooks_utils.other.jupyterlite.packages import install_packages

    await install_packages("api_examples")


### 1.2. Set parameters
Provide the INTERFACE_NAME to match the name of the structure in "uploads" folder for search. Left and right parts will be extracted based on substrate/film labels present in the generated interface.

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
INTERFACE_NAME = "Interface"  # To search for in "uploads" folder
LEFT_SIDE_PART = InterfacePartsEnum.SUBSTRATE
RIGHT_SIDE_PART = InterfacePartsEnum.FILM
INTERFACE_SYSTEM_NAME = None  # Used as tag to group the materials. Defaults to shorthand from the loaded interface name

IS_POLAR = False  # Whether the interface is polar, to adjust the VBO calculation method accordingly.

# 4. Workflow parameters
APPLICATION_NAME = "espresso"
WORKFLOW_SEARCH_TERM = "valence_band_offset.json"
MY_WORKFLOW_NAME = "VBO"

# 5. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set specific VBO parameters


In [ ]:
# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"  # "us" (ultrasoft), "nc" (norm-conserving), "paw"
FUNCTIONAL = "pbe"  # for gga: "pbe", "pbesol"; for lda: "pz"
MODEL_SUBTYPE = "gga"

# K-grid and k-path
SCF_KGRID = None  # e.g. [8, 8, 1]
KPATH = None  # e.g. [{"point": "G", "steps": 20}, {"point": "M", "steps": 20}]

# SCF diagonalization and mixing
DIAGONALIZATION = "david"  # "david" or "cg"
MIXING_BETA = 0.3

# Energy cutoffs
ECUTWFC = 40
ECUTRHO = 200


## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".


In [ ]:
from mat3ra.notebooks_utils.other.auth import authenticate

await authenticate()


### 2.2. Initialize API client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account


In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Create materials
### 3.1. Load interface from local folder


In [ ]:
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.python.other.material.visualize import visualize_materials as visualize

interface = load_material_from_folder(FOLDER, INTERFACE_NAME)

visualize(interface, repetitions=[1, 1, 1])
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")

### 3.2. Create materials from interface parts
Slabs are isolated based on labels, then labels removed as they are not compatible with Quantum ESPRESSO. The three materials (interface, left slab, right slab) are named and visualized.

In [ ]:
import re
from mat3ra.made.tools.modify import interface_get_part

interface_shorthand = re.match(r"^(.+?)\s", interface.name).group(1) if re.match(r"^(.+?)\s",
                                                                                 interface.name) else INTERFACE_NAME
interface_system_name = INTERFACE_SYSTEM_NAME or interface_shorthand
left_material = interface_get_part(interface, part=LEFT_SIDE_PART)
right_material = interface_get_part(interface, part=RIGHT_SIDE_PART)
interface_material = interface.clone()

left_material.basis.set_labels_from_list([])
right_material.basis.set_labels_from_list([])
interface_material.basis.set_labels_from_list([])

interface_material.name = f"{interface_system_name} Interface"
left_material.name = f"{interface_system_name} Left"
right_material.name = f"{interface_system_name} Right"

materials_by_role = {"interface": interface_material, "substrate": left_material, "film": right_material}
for role, material in materials_by_role.items():
    print(f"  {role}: {material.name}")
visualize(list(materials_by_role.values()), repetitions=[1, 1, 1], rotation="-90x")


### 3.3. Save materials


In [ ]:
from mat3ra.made.material import Material

saved_materials = {}
for role, material in materials_by_role.items():
    material_config = material.to_dict()
    material_config["name"] = material.name
    existing_tags = material_config.get("tags") or []
    material_config["tags"] = sorted(set([*existing_tags, interface_system_name]))
    saved = Material.create(client.materials.create(material_config, owner_id=ACCOUNT_ID))
    saved_materials[role] = saved
    print(f"  {role}: {saved.name} ({saved.id}) | tags={material_config['tags']}")


## 4. Configure workflow
### 4.1. Select application


In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")


### 4.2. Load workflow from Standata and preview it


In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.python.other.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)

workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)


### 4.3. Set model and its parameters (physics)


In [ ]:
from mat3ra.mode.model import Model
from mat3ra.standata.model_tree import ModelTreeStandata

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = Model.create(model_config)

for subworkflow in workflow.subworkflows:
    if subworkflow.application.name == APPLICATION_NAME:
        subworkflow.model = model


### 4.4. Modify method (computational parameters): k-grid, k-path, cutoffs, diagonalization, and mixing


In [ ]:
from mat3ra.wode.context.providers import (
    PlanewaveCutoffsContextProvider,
    PointsGridDataProvider,
    PointsPathDataProvider,
)


def set_pw_electrons_parameters(unit, diagonalization, mixing_beta):
    unit.replace_in_input_content(r"diagonalization\s*=\s*'[^']*'", f"diagonalization = '{diagonalization}'")
    unit.replace_in_input_content(r"mixing_beta\s*=\s*[-+0-9.eE]+", f"mixing_beta = {mixing_beta}")
    for input in unit.input:
        if isinstance(input, dict) and "content" in input:
            input["rendered"] = input["content"]
    return unit


for subworkflow in workflow.subworkflows:
    if subworkflow.application.name != APPLICATION_NAME:
        continue

    unit_names = [unit.name for unit in subworkflow.units]

    if SCF_KGRID is not None and "pw_scf" in unit_names:
        unit = subworkflow.get_unit_by_name(name="pw_scf")
        unit.add_context(PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data())
        subworkflow.set_unit(unit)

    if KPATH is not None and "pw_bands" in unit_names:
        unit = subworkflow.get_unit_by_name(name="pw_bands")
        unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).yield_data())
        subworkflow.set_unit(unit)

    cutoffs_context = PlanewaveCutoffsContextProvider(
        wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True
    ).yield_data()
    for unit_name in ["pw_scf", "pw_bands"]:
        if unit_name not in unit_names:
            continue
        unit = subworkflow.get_unit_by_name(name=unit_name)
        unit.add_context(cutoffs_context)
        unit = set_pw_electrons_parameters(unit, DIAGONALIZATION, MIXING_BETA)
        subworkflow.set_unit(unit)


### 4.5. Preview final workflow


In [ ]:
visualize_workflow(workflow)


## 5. Create the compute configuration
### 5.1. Get list of clusters


In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create compute configuration


In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


## 6. Create the job
### 6.1. Create job


In [ ]:
from mat3ra.notebooks_utils.python.other.api.helpers import create_job
from mat3ra.utils.object import dict_to_namespace_recursive
from mat3ra.notebooks_utils import display_JSON

materials = list(saved_materials.values())

job_name = f"{MY_WORKFLOW_NAME} {interface_system_name} {timestamp}"
workflow.name = job_name

print(f"Materials: {[m.id for m in materials]}")
print(f"Project: {project_id}")

job_response = create_job(
    api_client=client,
    materials=materials,
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")

display_JSON(job_response)


## 7. Submit the job and monitor the status


In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")


In [ ]:
from mat3ra.notebooks_utils.python.other.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)


## 8. Retrieve and visualize results
### 8.1. Valence Band Offset and average ESP profiles


In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.python.other.property.visualize import visualize_properties

job_data = client.jobs.get(job_id)
workflow = job_data["workflow"]

vbo_value_non_polar = client.properties.get_for_job(
    job_id,
    property_name=PropertyName.scalar.valence_band_offset.value,
)[0]
print(f"Workflow Valence Band Offset (VBO) value: {float(vbo_value_non_polar['value']):.3f} eV")

avg_esp_unit_ids = {}
for subworkflow in workflow["subworkflows"]:
    subworkflow_name = subworkflow["name"]
    for unit in subworkflow["units"]:
        result_names = [result["name"] for result in unit.get("results", [])]
        if "average_potential_profile" in result_names:
            avg_esp_unit_ids[subworkflow_name] = unit["flowchartId"]

ordered_names = [
    "BS + Avg ESP (Interface)",
    "BS + Avg ESP (interface left)",
    "BS + Avg ESP (interface right)",
]

avg_esp_results = {}
for subworkflow_name in ordered_names:
    unit_id = avg_esp_unit_ids[subworkflow_name]
    avg_esp_data = client.properties.get_for_job(
        job_id,
        property_name="average_potential_profile",
        unit_id=unit_id,
    )[0]
    avg_esp_results[subworkflow_name] = avg_esp_data
    visualize_properties(avg_esp_data, title=subworkflow_name)



### 8.2. Polar-only notebook post-process
If `IS_POLAR = True`, use the slab boundaries from the notebook materials and fit the ESP profiles directly in the notebook.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.python.other.api.helpers import get_property_holder_for_job, update_property_holder_value


def get_region_indices(x_data, x_min, x_max):
    mask = (x_data >= x_min) & (x_data <= x_max)
    indices = np.where(mask)[0]
    if len(indices) == 0:
        return 0, len(x_data)
    return indices[0], indices[-1] + 1


def fit_and_average(x_data, y_data, start_idx, end_idx):
    x_region = x_data[start_idx:end_idx]
    y_region = y_data[start_idx:end_idx]
    if len(x_region) < 2:
        avg = float(np.mean(y_region)) if len(y_region) > 0 else 0.0
        return avg, 0.0, avg
    slope, intercept, _, _, _ = linregress(x_region, y_region)
    x_mid = (x_region[0] + x_region[-1]) / 2.0
    avg_value = slope * x_mid + intercept
    return float(avg_value), float(slope), float(intercept)


interface_profile = avg_esp_results["BS + Avg ESP (Interface)"]
left_profile = avg_esp_results["BS + Avg ESP (interface left)"]
right_profile = avg_esp_results["BS + Avg ESP (interface right)"]

interface_material.to_cartesian()
left_material.to_cartesian()
right_material.to_cartesian()

interface_z = sorted(coord[2] for coord in interface_material.basis.coordinates.values)
n_left = len(left_material.basis.elements.values)
z_min_1 = interface_z[0]
z_max_1 = interface_z[n_left - 1]
z_min_2 = interface_z[n_left]
z_max_2 = interface_z[-1]

print(f"Detected slab 1 boundaries: z = {z_min_1:.3f} to {z_max_1:.3f} A")
print(f"Detected slab 2 boundaries: z = {z_min_2:.3f} to {z_max_2:.3f} A")

X = np.array(interface_profile["xDataArray"])
Y = np.array(interface_profile["yDataSeries"][1])
X_left = np.array(left_profile["xDataArray"])
Y_left = np.array(left_profile["yDataSeries"][1])
X_right = np.array(right_profile["xDataArray"])
Y_right = np.array(right_profile["yDataSeries"][1])

slab1_start, slab1_end = get_region_indices(X, z_min_1, z_max_1)
slab2_start, slab2_end = get_region_indices(X, z_min_2, z_max_2)
slab1_start_left, slab1_end_left = get_region_indices(X_left, z_min_1, z_max_1)
slab2_start_right, slab2_end_right = get_region_indices(X_right, z_min_2, z_max_2)

Va_interface, slope_a, intercept_a = fit_and_average(X, Y, slab1_start, slab1_end)
Vb_interface, slope_b, intercept_b = fit_and_average(X, Y, slab2_start, slab2_end)
Va_bulk_left, _, _ = fit_and_average(X_left, Y_left, slab1_start_left, slab1_end_left)
Vb_bulk_right, _, _ = fit_and_average(X_right, Y_right, slab2_start_right, slab2_end_right)

vbo_value_polar = (Vb_interface - Va_interface) - (Vb_bulk_right - Va_bulk_left)

print(f"Interface ESP slab 1: {Va_interface:.3f} eV")
print(f"Interface ESP slab 2: {Vb_interface:.3f} eV")
print(f"Bulk ESP left: {Va_bulk_left:.3f} eV")
print(f"Bulk ESP right: {Vb_bulk_right:.3f} eV")
print(f"Fit slope slab 1: {slope_a:.6f} eV/A")
print(f"Fit slope slab 2: {slope_b:.6f} eV/A")
print(f"Polar post-process VBO: {vbo_value_polar:.3f} eV")

### 8.3. Plot the ESP profiles with slab regions and fits

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(X, Y, label="Macroscopic Average Potential", linewidth=2)
plt.axvspan(z_min_1, z_max_1, color="red", alpha=0.2, label="Slab 1 Region")
plt.axvspan(z_min_2, z_max_2, color="blue", alpha=0.2, label="Slab 2 Region")

if slab1_end > slab1_start:
    x_fit1 = X[slab1_start:slab1_end]
    y_fit1 = slope_a * x_fit1 + intercept_a
    plt.plot(x_fit1, y_fit1, color="darkred", linestyle="--", linewidth=2, label="Fit Slab 1")

if slab2_end > slab2_start:
    x_fit2 = X[slab2_start:slab2_end]
    y_fit2 = slope_b * x_fit2 + intercept_b
    plt.plot(x_fit2, y_fit2, color="darkblue", linestyle="--", linewidth=2, label="Fit Slab 2")

plt.axhline(Va_interface, color="red", linestyle=":", linewidth=2, label=f"Avg ESP Slab 1 = {Va_interface:.3f} eV")
plt.axhline(Vb_interface, color="blue", linestyle=":", linewidth=2, label=f"Avg ESP Slab 2 = {Vb_interface:.3f} eV")
plt.xlabel("z-coordinate (A)", fontsize=12)
plt.ylabel("Macroscopic Average Potential (eV)", fontsize=12)
plt.title(f"Polar Interface VBO = {vbo_value_polar:.3f} eV", fontsize=14, fontweight="bold")
plt.legend(loc="best", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 8.4. Update the VBO property with the polar post-process result

In [ ]:
if IS_POLAR:
    vbo_property_holder = get_property_holder_for_job(
        client,
        job_id,
        PropertyName.scalar.valence_band_offset.value,
    )
    update_property_holder_value(client, vbo_property_holder["_id"], vbo_value_polar)
    print(f"Persisted displayed Valence Band Offset (VBO) value: {vbo_value_polar:.3f} eV")

